In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time
import math
import platform

In [2]:
try:
    from playsound import playsound
    _HAS_PLAYSOUND = True
except Exception:
    _HAS_PLAYSOUND = False

# Windows beep fallback
_IS_WINDOWS = platform.system().lower().startswith('win')
if _IS_WINDOWS:
    try:
        import winsound
        _HAS_WINSOUND = True
    except Exception:
        _HAS_WINSOUND = False
else:
    _HAS_WINSOUND = False

In [3]:
# Configuration (tweak these)
EAR_THRESHOLD = 0.23       # eye aspect ratio below which eye is considered "closed"
EAR_CONSEC_FRAMES = 20     # number of consecutive frames eye must be below thresh to trigger drowsiness (increase for robustness)
MAR_THRESHOLD = 0.6        # mouth aspect ratio above which we consider a yawn (tweak per camera/distances)
YAWN_CONSEC_FRAMES = 15    # consecutive frames mouth open to count as yawn

ALARM_ON = True            # whether to play alarm sound on drowsiness
ALARM_PATH = "alarm.wav"   # path to alarm wav file if using playsound fallback

# Camera index (0 is default webcam)
CAM_INDEX = 0

In [4]:
# Mediapipe setup
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

# Face mesh options
FACE_MESH_MAX_NUM = 1
MIN_DETECTION_CONFIDENCE = 0.5
MIN_TRACK_CONFIDENCE = 0.5

In [5]:
# Cell 2: helper functions

def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def eye_aspect_ratio(landmarks, eye_indices, img_w, img_h):
    """
    landmarks: list of normalized landmark points from mediapipe (x,y,z)
    eye_indices: list of 6 indices for the eye [p1,p2,p3,p4,p5,p6]
    returns EAR (float)
    """
    coords = [(int(landmarks[i].x * img_w), int(landmarks[i].y * img_h)) for i in eye_indices]
    p1, p2, p3, p4, p5, p6 = coords
    # EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)
    A = euclidean(p2, p6)
    B = euclidean(p3, p5)
    C = euclidean(p1, p4)
    if C == 0:
        return 0.0
    ear = (A + B) / (2.0 * C)
    return ear, coords

def mouth_aspect_ratio(landmarks, mouth_indices, img_w, img_h):
    """
    Simple MAR using top and bottom lip and left-right mouth corners.
    mouth_indices = [upper_lip, lower_lip, left_mouth, right_mouth]
    returns MAR (float) and coords
    """
    coords = [(int(landmarks[i].x * img_w), int(landmarks[i].y * img_h)) for i in mouth_indices]
    up, down, left, right = coords
    vertical = euclidean(up, down)
    horizontal = euclidean(left, right)
    if horizontal == 0:
        return 0.0, coords
    mar = vertical / horizontal
    return mar, coords

def sound_alarm():
    """
    Cross-platform alarm:
     - On Windows tries winsound.Beep
     - Else tries playsound(alarm.wav) if available and path exists
    """
    if not ALARM_ON:
        return
    try:
        if _HAS_WINSOUND:
            # beep frequency 2000Hz, duration 700ms
            winsound.Beep(2000, 700)
        elif _HAS_PLAYSOUND:
            # playsound is blocking; runs quickly for short file
            playsound(ALARM_PATH)
        else:
            # fallback: print bell character (may or may not sound)
            print('\a')  # system beep
    except Exception as e:
        # don't crash main loop because of alarm error
        print("Alarm error:", e)

In [6]:
# Cell 3: landmark index sets for MediaPipe Face Mesh

# These indices are commonly used with mediapipe face mesh for the eye landmarks:
LEFT_EYE_IDX  = [33, 160, 158, 133, 153, 144]   # p1...p6
RIGHT_EYE_IDX = [263, 387, 385, 362, 380, 373]  # p1...p6

# Mouth indices: upper lip, lower lip, left corner, right corner
# Using common mediapipe points
MOUTH_IDX = [13, 14, 78, 308]

In [7]:
# Cell 4: main real-time loop. Run this cell to start webcam drowsiness detection.

# Counters
COUNTER_EAR = 0
COUNTER_YAWN = 0
ALARM_STATE = False

# Video capture
cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open webcam (index {CAM_INDEX}).")

# MediaPipe face mesh instance
with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=FACE_MESH_MAX_NUM,
    refine_landmarks=True,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
    min_tracking_confidence=MIN_TRACK_CONFIDENCE
) as face_mesh:

    prev_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read frame from webcam. Exiting.")
            break

        # Flip for natural view
        frame = cv2.flip(frame, 1)
        img_h, img_w = frame.shape[:2]

        # Convert to RGB for mediapipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process
        results = face_mesh.process(rgb_frame)

        # Default overlay text
        status_text = "Normal"

        if results.multi_face_landmarks:
            # Use first face only
            landmarks = results.multi_face_landmarks[0].landmark

            # EAR left & right
            ear_left, left_coords = eye_aspect_ratio(landmarks, LEFT_EYE_IDX, img_w, img_h)
            ear_right, right_coords = eye_aspect_ratio(landmarks, RIGHT_EYE_IDX, img_w, img_h)
            ear = (ear_left + ear_right) / 2.0

            # Draw eye contours (for visualization)
            for (x, y) in left_coords + right_coords:
                cv2.circle(frame, (x, y), 1, (0,255,0), -1)

            # Check EAR threshold
            if ear < EAR_THRESHOLD:
                COUNTER_EAR += 1
            else:
                # reset when open
                if COUNTER_EAR >= EAR_CONSEC_FRAMES:
                    # optional: log blink/drowsiness event
                    pass
                COUNTER_EAR = 0
                ALARM_STATE = False

            # Drowsiness triggered
            if COUNTER_EAR >= EAR_CONSEC_FRAMES:
                status_text = "DROWSY!"
                if not ALARM_STATE:
                    ALARM_STATE = True
                    # play alarm asynchronously (we call but it may block small time)
                    # For a non-blocking approach, you can spawn a thread if needed.
                    try:
                        # call sound_alarm (may block briefly)
                        sound_alarm()
                    except Exception as e:
                        print("Alarm play failed:", e)

                # Visual alert
                cv2.putText(frame, "DROWSINESS ALERT!", (30,80), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)

            # MAR - yawn detect
            mar, mouth_coords = mouth_aspect_ratio(landmarks, MOUTH_IDX, img_w, img_h)
            # draw mouth points
            for (x, y) in mouth_coords:
                cv2.circle(frame, (x, y), 2, (255,0,0), -1)

            if mar > MAR_THRESHOLD:
                COUNTER_YAWN += 1
            else:
                COUNTER_YAWN = 0

            if COUNTER_YAWN >= YAWN_CONSEC_FRAMES:
                status_text = "YAWNING"
                cv2.putText(frame, "Yawn Detected", (30,130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,165,255), 2)
                # you can sound a different alarm or visual notification

            # Draw EAR/MAR on frame
            cv2.putText(frame, f"EAR: {ear:.3f}", (30,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            cv2.putText(frame, f"MAR: {mar:.3f}", (200,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        else:
            cv2.putText(frame, "No face detected", (30,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)

        # Show status
        cv2.putText(frame, f"Status: {status_text}", (30,60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0) if status_text=="Normal" else (0,0,255), 2)

        # Show FPS
        cur_time = time.time()
        fps = 1.0 / (cur_time - prev_time) if (cur_time - prev_time) > 0 else 0.0
        prev_time = cur_time
        cv2.putText(frame, f"FPS: {int(fps)}", (img_w - 120, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

        # Display
        cv2.imshow("Driver Drowsiness Detection", frame)

        # Quit with 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Cleanup
cap.release()
cv2.destroyAllWindows()


C:\Users\User\AppData\Roaming\Python\Python311\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
